In [2]:
import tomotopy as tp
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
from operator import itemgetter
import matplotlib.pyplot as plt
import pandas as pd
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
import re
import openai
from googletrans import Translator

In [3]:
def get_topics(line):
    line=line.replace('\'','')
    line=line.split("[")[1]
    line=line.split("]")[0]
    return line.split(", ")

In [4]:
def get_word_embedding_based_similarity(topic1, topic2):
    similarity = 0
    for i in topic_dict.get(topic1)[:top_n]:
        for j in topic_dict.get(topic2)[:top_n]:
            similarity+= word2vec_model.wv.similarity(i,j)
    return similarity/(top_n*top_n)

In [5]:
def get_similarity(parent_dict):
    similarity = 0
    cnt = 0
    for parent in parent_dict:
        for child in parent_dict.get(parent):
            similarity+=get_word_embedding_based_similarity(parent,child)
            cnt+=1
    return similarity/cnt

In [6]:
def get_diveristy(parent_dict):
    diversity = 0
    cnt = 0
    for parent in parent_dict:
        siblings = parent_dict.get(parent)
        for i in range(len(siblings)):
            for j in range(i):
                diversity+=(1-get_word_embedding_based_similarity(siblings[i],siblings[j]))
                cnt+=1
    return diversity / cnt

In [7]:
def get_coherence(topic_dict):
    topics=[]
    for topic in topic_dict:
        topics.append(topic_dict.get(topic))
        
    cm = CoherenceModel(topics = topics,
                       #corpus = corpus,
                       dictionary = dictionary,
                       texts = texts,
                       coherence = 'c_npmi')

    return cm.get_coherence()

In [8]:
top_n = 20

In [9]:
word2vec_model = Word2Vec.load('../models/word2vec.model')

In [10]:
input_data = pd.read_feather('../results/dataset.ftr')
texts =input_data.okt

dictionary = Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]


In [11]:
score_list = []
topic_dict_list = []
parent_dict_list = []
for sample_num in range(5):
    file=open(f'../Datasets/hierarchical_struture{sample_num}.txt', 'r')
    data = file.readlines()
    
    
    num_depth0 = 0
    num_depth1 = 0
    num_depth2 = 0
    for line in data:
        if '\t' not in line: num_depth0+=1
        elif '\t\t'not in line: num_depth1+=1
        else: num_depth2+=1
    root = num_depth0
    num_depth1 += num_depth0
    num_depth2 += num_depth2
    
    
    topic_dict={}
    parent_dict={}

    for line in data:
        # depth0
        if '\t' not in line:
            num_depth0 -=1
            topic_dict[num_depth0] = get_topics(line)

        # depth1
        elif '\t\t'not in line:
            num_depth1 -=1
            topic_dict[num_depth1] = get_topics(line)
            if parent_dict.get(num_depth0) is None :
                parent_dict[num_depth0] = [num_depth1]
            else :
                temp = parent_dict.get(num_depth0)
                temp.append(num_depth1)
                parent_dict[num_depth0] = temp

        # depth2
        else:
            num_depth2 -=1
            topic_dict[num_depth2] = get_topics(line)
            if parent_dict.get(num_depth1) is None :
                parent_dict[num_depth1] = [num_depth2]
            else :
                temp = parent_dict.get(num_depth1)
                temp.append(num_depth2)
                parent_dict[num_depth1] = temp
   
    topic_dict_list.append(topic_dict)
    parent_dict_list.append(parent_dict)
    similarity = get_similarity(parent_dict)
    diversity = get_diveristy(parent_dict)
    coherence = get_coherence(topic_dict)
    score = (0.3 *(similarity) + 0.3*(diversity)+0.4*(coherence))
    temp = [similarity,diversity,coherence,score]
    score_list.append(temp)

In [12]:
score_list

[[0.7627730537183355,
  0.1838707780190131,
  -0.41747968319059703,
  0.11700127624496576],
 [0.6380711820162833,
  0.31038051727848753,
  -0.43897627593580585,
  0.10894499941410887],
 [0.6687965047678778,
  0.21586115633802752,
  -0.42213438955137256,
  0.09654354251122257],
 [0.7502105800360442,
  0.25251598426699634,
  -0.43603953014356056,
  0.12640215723348797],
 [0.6911993181581297,
  0.31996337773899236,
  -0.4306936745795835,
  0.1310713389373032]]

In [13]:
print(((score_list[0][0])+(score_list[1][0])+(score_list[2][0])+(score_list[3][0])+(score_list[4][0]))/5)
print(((score_list[0][1])+(score_list[1][1])+(score_list[2][1])+(score_list[3][1])+(score_list[4][1]))/5)
print(((score_list[0][2])+(score_list[1][2])+(score_list[2][2])+(score_list[3][2])+(score_list[4][2]))/5)
print(((score_list[0][3])+(score_list[1][3])+(score_list[2][3])+(score_list[3][3])+(score_list[4][3]))/5)

0.702210127739334
0.25651836272830336
-0.4290647106801839
0.11599266286821767


# Interpretation

In [14]:
topic_dict = topic_dict_list[np.argmax(score_list)]
parent_dict = parent_dict_list[np.argmax(score_list)]

In [15]:
topics = []
for i in range(root):
    print(f"topic{i} : ",topic_dict.get(i))
    if parent_dict.get(i) is not None:
        for d1_topic in parent_dict.get(i):
            print(f"  topic{d1_topic} : ",topic_dict.get(d1_topic))
            if parent_dict.get(d1_topic) is not None:
                for d2_topic in parent_dict.get(d1_topic):
                    print(f"    topic{d2_topic} : ",topic_dict.get(d2_topic))
                print()

topic0 :  ['힘차다', '방어율', '방주', '방인', '방위', '방울', '방역패스', '방역', '방어', '방법', '방안', '방심', '방식', '방송사', '방송', '방사선', '방증', '방지', '방짜', '방출']
topic1 :  ['힘차다', '방어율', '방주', '방인', '방위', '방울', '방역패스', '방역', '방어', '방법', '방안', '방심', '방식', '방송사', '방송', '방사선', '방증', '방지', '방짜', '방출']
topic2 :  ['힘차다', '방어율', '방주', '방인', '방위', '방울', '방역패스', '방역', '방어', '방법', '방안', '방심', '방식', '방송사', '방송', '방사선', '방증', '방지', '방짜', '방출']
topic3 :  ['델타', '베타', '옮아', '변이', '산술', '전염', '저항력', '변인', '무력', '아데노바이러스', '항원', '바이러스', '단백질', '유전자', '스파이크', '변종', '벡터', '중화항체', '말림', '브라질']
  topic13 :  ['델타', '옮아', '베타', '산술', '변이', '저항력', '변인', '전염', '무력', '아데노바이러스', '항원', '스파이크', '유전자', '단백질', '브라질', '바이러스', '매년', '중화항체', '말림', '벡터']
  topic12 :  ['가능성', '위험', '비율', '폴리에틸렌', '예방', '경증', '변종', '낮다', '감염률', '치료', '오미크론', '예방률', '사망자', '바이러스', '의하다', '중증', '성률', '효과', '감마', '초기']
    topic36 :  ['변종', '바이러스', '예방', '오미크론', '감염률', '예방률', '벡터', '변이', '말림', '낮다', '람다', '감마', '효과', '방식', '단백질', '치명', '중증', '방어율', '초기', '중화항체']
  